In [24]:
#state banaune 

from typing import TypedDict , Annotated, List
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage

class ChatState(TypedDict):
  messages : Annotated[List[BaseMessage], add_messages]
  
#add_messages Reducer add gareko so that Euta Node le add gareko message le old message lai replace nagaros BUT append garos

#Annotated[] me hamile data ko Types and Extra Information rakhna sakincha

#HumanMessage , SystemMessage , AIMessage , ToolMessage sabaile BaseMessage lai Inherit gareko huncha

In [25]:
#llm model

import os 
from dotenv import load_dotenv
load_dotenv()

MODEL_API_KEY = os.getenv("MODEL_API_KEY")
MODEL_NAME  = os.getenv("MODEL_NAME")
MODEL_PROVIDER = os.getenv("MODEL_PROVIDER")

#langchain import
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate

system_prompt = """Answer the user's query in one short, clear, and accurate response. Use simple language, avoid unnecessary details and multiple separate answers, and focus only on what the user asks."""

prompt_template = ChatPromptTemplate.from_messages(
  [
    ("system",system_prompt),
    ("human","{user_query}")
  ]
)

llm = init_chat_model(
  model=MODEL_NAME,
  model_provider=MODEL_PROVIDER,
  api_key = MODEL_API_KEY
)

In [26]:
#RAM ma store garna lai 
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

In [27]:
#node banaune 

#Node -> bhaneko chai simply Python Function ho 
#jasle State ko value excess garcha and state ko value change garcha


# Rule 1: Node एउटा Python function हुनुपर्छ
# Rule 2: State access गर्नुपर्छ भने parameter मा लिनुपर्छ
# Rule 3: Update गर्नुपर्ने data dictionary मा return गर्नुपर्छ
# Rule 4: Return गरेको data State सँग मिल्नुपर्छ
# Rule 5: Node लाई Graph मा register गर्नुपर्छ


def chat_node(state : ChatState):
  
  # user ko query line
  user_question = state["messages"]
  
  #response generate garne
  chain = prompt_template | llm
  
  response = chain.invoke({
    "user_query" : user_question
  })
  
  #response lai state ma store garne
  #dictionary return garne
  return {
    "messages" : [response]
  }

In [28]:
#graph builder banaune
from langgraph.graph import StateGraph, START, END

builder = StateGraph(ChatState)

#node lai graph ma add garne i.e python function lai node banaune
builder.add_node("chat_node",chat_node)

#abo graph banaune
builder.add_edge(START,"chat_node")
builder.add_edge("chat_node",END)

#abo graph banaune i.e compile garne

#Graph ready garda checkpoint sanga connect garne
graph = builder.compile(checkpointer=checkpointer)

In [29]:
#Config banaune jun chai Dictionary ho

thread_id = "ujjwal02"

config = {
  "configurable" : {
    "thread_id" : thread_id
  }
}

In [ ]:
# #user ko query line
# user_query = input("ASK : ")

# #langchain humanmessage 
# from langchain_core.messages import HumanMessage

# result = graph.invoke({
#   "messages" : [HumanMessage(content=user_query)]
# })

# print(result["messages"])

In [31]:
# #langchain humanmessage 
from langchain_core.messages import HumanMessage

while True:
  user_query = input("ASK (Type 'exit' to Exit): ")
  
  if user_query.strip().lower() == "exit":
    break
  
  print(f"USER : {user_query}")
  result = graph.invoke(
    {
      "messages" : [HumanMessage(content=user_query)]
    }
  ,config=config)
  
  print(f"AI : {result["messages"][-1].content}")

USER : Hi my name is ujjwal
AI : Hello Ujjwal! How can I help you today?
USER : what is mine name?
AI : Your name is Ujjwal.
